# Piper-Neo Studio · Instalación y prueba limpia en Kaggle

Este cuaderno importa el proyecto **con historial Git** (si usas el ZIP completo) o clona el repositorio al que lo hayas subido. Después instala las dependencias **fijadas**, verifica la GPU NVIDIA, ejecuta pruebas de regresión y abre Gradio.

**Antes de iniciar:** en Kaggle, activa **Settings → Accelerator → GPU** y **Internet = ON**. Selecciona fuente **ZIP** o **GitHub** en la primera celda. Para una instalación nueva, espera varios minutos y reserva unos **5 GB para dependencias**, además del espacio de los datos.

**Importante:** Este ZIP **no incluye datasets, checkpoints ni tokens**. Para reanudar un entrenamiento tendrás que preparar/preprocesar el dataset de la misma voz y descargar el último checkpoint válido desde HF mediante la pestaña **3. Entrenamiento**. El repositorio privado HF requiere permisos de lectura y escritura según la operación.

El servidor Gradio público sin usuario/contraseña está abierto a cualquier persona que tenga su URL. Si vas a introducir tokens, configura credenciales de Gradio en los secretos de Kaggle o no compartas públicamente el servidor.

In [ ]:
# 1. ELIGE de dónde importar el código
SOURCE = "GitHub"   # "GitHub" o "ZIP"

# Para GitHub: URL de TU repositorio (debe incluir la rama y commits que subiste).
REPO_URL = "https://github.com/TU_USUARIO/Piper-Neo.git"
BRANCH = "feature/resilient-finetune-studio"

# Para ZIP: adjunta el ZIP como Dataset/Input de Kaggle e indica su ruta.
ZIP_PATH = "/kaggle/input/tu-dataset/Piper-Neo-Studio-GIT.zip"

# Si usas la rama main de tu nuevo repo en lugar de feature, cambia BRANCH = "main".
ROOT = "/kaggle/working/Piper-Neo"

In [ ]:
import os, pathlib, subprocess, zipfile, sys
from pathlib import Path

project_dir = Path(ROOT)
if project_dir.exists() and (project_dir / ".git").is_dir():
    print("Proyecto ya disponible; no se reemplaza:", project_dir)
elif project_dir.exists():
    raise RuntimeError(f"{project_dir} ya existe pero no parece un repo Git. No se borra.")
elif SOURCE.strip().lower() == "github":
    if "TU_USUARIO" in REPO_URL:
        raise ValueError("Cambia REPO_URL por la URL de TU repositorio de GitHub.")
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                    REPO_URL, str(project_dir)], check=True)
elif SOURCE.strip().lower() == "zip":
    archive = Path(ZIP_PATH)
    if not archive.is_file():
        raise FileNotFoundError(f"ZIP no encontrado: {archive}. Revisa 'Add Input'.")
    with zipfile.ZipFile(archive) as bundle:
        members = bundle.namelist()
        if not any(m.startswith("Piper-Neo/.git/") for m in members):
            raise ValueError("El ZIP no contiene el historial Git esperado.")
        for m in members:
            path = Path(m)
            if path.is_absolute() or ".." in path.parts or not m.startswith("Piper-Neo/"):
                raise ValueError(f"Ruta insegura en el archivo: {m!r}")
        bundle.extractall(Path("/kaggle/working"))
else:
    raise ValueError("SOURCE debe ser GitHub o ZIP")

commit = subprocess.check_output(["git", "-C", ROOT, "log", "-1", "--format=%h %s"], text=True).strip()
branch = subprocess.check_output(["git", "-C", ROOT, "branch", "--show-current"], text=True).strip()
print("Proyecto:", ROOT, "\nRama:", branch, "\nÚltimo commit:", commit)
print("Historial Git disponible:", (project_dir / ".git").is_dir())

## 2. Instalar dependencias compatibles y la extensión nativa

El script oficial del proyecto instala **Python 3.10 aislado** usando `uv`, PyTorch de la serie compatible con Piper Train, Gradio, ONNX y compila `monotonic_align`. No instala datasets ni sobrescribe checkpoints. Requiere Internet en Kaggle. **Este paso no es un ejecutable precompilado**; el runtime portable es una entrega futura distinta.

In [ ]:
import subprocess, os
subprocess.run(
    ["bash", str(project_dir / "script/setup_kaggle_finetune.sh")],
    cwd=str(project_dir),
    check=True,
    env={**os.environ, "LD_LIBRARY_PATH": "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")},
)
PY = project_dir / "src/python/.venv/bin/python"
if not PY.is_file():
    raise RuntimeError("No se creó Python 3.10 del proyecto: " + str(PY))
print("Entorno instalado:", PY)

## 3. Confirmar GPU y hacer pruebas de calidad

Esta celda no inicia fine-tune: solo prueba imports CUDA, muestra la GPU (Tesla T4 u otra NVIDIA compatible) y corre los tests del respaldo HF, reanudación e interfaz. Si CUDA da `False`, revisa Accelerator GPU en Kaggle antes de entrenar.

In [ ]:
import subprocess, os
env = {
    **os.environ,
    "LD_LIBRARY_PATH": "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", ""),
}
gpu_code = """
import torch
from piper_train.gradio_finetune import build_ui
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("Gradio components:", len(build_ui().blocks))
"""
subprocess.run([str(PY), "-c", gpu_code], cwd=str(project_dir / "src/python"), env=env, check=True)
subprocess.run(
    [str(PY), "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py", "-v"],
    cwd=str(project_dir / "src/python"), env=env, check=True,
)
print("Verificaciones completadas.")

## 4. Configurar secretos (opcional, recomendado)

En **Kaggle → Add-ons → Secrets**, puedes agregar:

- `HF_TOKEN` para repositorios de Hugging Face.
- `PIPER_STUDIO_AUTH_USER` y `PIPER_STUDIO_AUTH_PASSWORD` para proteger el enlace de Gradio.

Los secretos se pasan únicamente al proceso servidor y no se imprimen ni se guardan en el notebook. Si dejas user/password sin definir, Gradio no pide contraseña.

In [ ]:
import os
from pathlib import Path

server_env = {
    **os.environ,
    "PYTHONUNBUFFERED": "1",
    "LD_LIBRARY_PATH": "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", ""),
}
try:
    from kaggle_secrets import UserSecretsClient
    secret_client = UserSecretsClient()
    for key in ("HF_TOKEN", "PIPER_STUDIO_AUTH_USER", "PIPER_STUDIO_AUTH_PASSWORD"):
        try:
            value = secret_client.get_secret(key)
            if value:
                server_env[key] = value
        except Exception:
            pass  # El secreto es opcional.
except ImportError:
    print("kaggle_secrets no está disponible (fuera de Kaggle).")

print("Token HF configurado:", bool(server_env.get("HF_TOKEN")))
print("Protección Gradio configurada:", bool(
    server_env.get("PIPER_STUDIO_AUTH_USER") and
    server_env.get("PIPER_STUDIO_AUTH_PASSWORD")
))

## 5. Abrir Gradio con 5 pestañas

La celda inicia el servidor en un **proceso independiente** y muestra la URL temporal `gradio.live`. No vuelvas a ejecutarla si ya está activo. Los archivos y progreso están en `/kaggle/working/piper_finetune`, no en el navegador. Kaggle puede cerrar su runtime; descarga respaldos HF regularmente.

**Pestañas:** Dataset · Preprocess · Entrenamiento/Resume HF · Estado/Respaldo incremental · Inferencia/ONNX.

In [ ]:
import subprocess, time, re
from pathlib import Path

PORT = 7860
web_log = project_dir / "studio_kaggle.log"
existing = globals().get("studio_process")
if existing is not None and existing.poll() is None:
    print("Gradio ya corre, PID", existing.pid)
else:
    handle = web_log.open("a", encoding="utf-8")
    studio_process = subprocess.Popen(
        [str(PY), "-u", "-m", "piper_train.gradio_finetune",
         "--share", "--port", str(PORT)],
        cwd=str(project_dir / "src/python"),
        env=server_env,
        stdout=handle, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    handle.close()
    print("Gradio iniciado, PID:", studio_process.pid)

url = None
for _ in range(90):
    content = web_log.read_text(encoding="utf-8", errors="replace") if web_log.exists() else ""
    matches = re.findall(r"https://[a-zA-Z0-9.-]+\.gradio\.live", content)
    if matches:
        url = matches[-1]
        break
    if studio_process.poll() is not None:
        raise RuntimeError("Gradio terminó inesperadamente:\n" + content[-4000:])
    time.sleep(1)
print("URL pública:", url or "Aún generándose: revisa studio_kaggle.log")
print("El enlace requiere contraseña solamente si configuraste ambos secretos de acceso.")

## 6. Revisar servidor sin reiniciar el entrenamiento

Puedes ejecutar esta celda cuando quieras. Si Gradio sigue corriendo, recarga su URL. Si debes reconstruir el runtime, usa `Resume de corrida`, prepara el mismo dataset y usa **Hugging Face: último válido** para descargar el checkpoint y recuperar época/optimizadores.

In [ ]:
from pathlib import Path
status = globals().get("studio_process")
print("Servidor:", "activo" if status and status.poll() is None else "no activo en este kernel")
if web_log.exists():
    print("\n".join(web_log.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]))
print("Datos persistentes:", Path("/kaggle/working/piper_finetune").exists())